In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display, Markdown
import pandas as pd
import time
from rag_flow import RAGFlow


/Users/vilya/Desktop/capstone_project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-05 20:03:05,406 | INFO | No device provided, using mps
2026-10-05 20:03:05,545 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-10-05 20:03:05,546 | INFO | No modules.json found for cross-encoder/ms-marco-MiniLM-L6-v2, initializing a new CrossEncoder model.
2026-10-05 20:03:05,635 | INFO | HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-10-05 20:03:05,728 | INFO | HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-10-05 20:03:05,826 | INFO | HTTP Request: HEAD https://huggingface.co/cross-enc

In [2]:
def load_model():
    model_name = "Qwen/Qwen3-8B"
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto",
        local_files_only=True #swith it off to download from hugging face if you don't have it locally
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
    return {"model": model, "tokenizer": tokenizer}


In [3]:
model_and_tokenizer = load_model()
model = model_and_tokenizer["model"]
tokenizer = model_and_tokenizer["tokenizer"]

Loading weights: 100%|██████████| 399/399 [00:01<00:00, 313.62it/s]


In [4]:
params = {
    "model": model,
    "tokenizer": tokenizer,
    "web_query_max_new_t": 50,
    "titles_limit": 5,
    "top_k_titles": 5,
    "chunk_size": 200,
    "chunk_overlap_fraction": 0.2,
    "top_k_chunks": 5,
    "answer_max_new_t": 500
}


rag = RAGFlow(params)

#answer = rag.run("How does retrieval-augmented generation reduce hallucinations in large language models??")


## Evaluation
### System level:
- Token usage per prompt.
- End-to-End Latency
- Context recall (relevant answers / (relevant answers + irrelevant answers). To determine whether an answer is relevant, I will use a threshold of similarity between the actual answer and the correct answer. For example, the cosine similarity is 0.9 then the answer is marked relevant.
- Context precision (relevant answers / total answers)
- Faithfulness (Is the answer strictly true to the retrieved context?" (Checks for hallucination) I am going to use RAGAS library ( if I have no issues configuring it and a LLM-as-judge)
### Retrieval level
- Context precision/ recall @ k
- Mean reciprocal rank (MRR)
- Normalized Discounted Cumulative Gain (NDCG)  how well top-k is sorted




## To do:
- generate ground truth
- test and record the results
- download 1 -2 more models and test the same flow with them. Chose the one that performed best
- With best model, fine-tune the system:
    - test different numbers of searched titles
    - test different number of downloaded articles
    - try different chunk sizes
    - try different number of top k in the chunk ranker
    - try a set of promot augmentation.

In [5]:
sys_evaluation_df = {
    "mean_latency": "",
    "mean_token_usage":"",
    "context_recall":"",
    "context_precision":"",
    "mean_faithfulness":""
}

In [6]:
golden_df = pd.read_csv("golden_dataset.csv")

In [7]:
results = []

for _, row in golden_df.iterrows():
    golden_id = row["golden_id"]
    question = row["question"]
    start = time.perf_counter()
    answer = rag.run(question)
    latency = time.perf_counter() - start
    results.append({
        "golden_id": golden_id,
        "answer": answer,
        "latency": latency
    })

evaluation_df = pd.DataFrame(results)
evaluation_df.to_csv("system_eval_results.csv")


2026-10-05 20:03:10,847 | INFO | user prompt: How does confidence supervision cut tokens by 25% and preserve accuracy Gemma, Qwen, Nemotron, GPT‑OSS?, web query: confidence supervision token reduction model efficiency
2026-10-05 20:03:22,905 | WARNING | arXiv timeout/no results for query: confidence supervision token reduction model efficiency
2026-10-05 20:03:23,897 | INFO | user prompt: How does confidence supervision cut tokens by 25% and preserve accuracy Gemma, Qwen, Nemotron, GPT‑OSS?, web query: confidence supervision token reduction model efficiency
2026-10-05 20:03:23,898 | INFO | Retry query: confidence supervision token reduction model efficiency
2026-10-05 20:03:35,355 | WARNING | arXiv timeout/no results for query: confidence supervision token reduction model efficiency
Batches: 100%|██████████| 1/1 [00:00<00:00, 41.42it/s]
2026-10-05 20:03:35,471 | ERROR | RAG failed: Invalid URL 'timeout': No scheme supplied. Perhaps you meant https://timeout?
2026-10-05 20:03:36,532 | I

In [36]:
pd.set_option("display.max_colwidth", None)
results = pd.DataFrame(results)
results.head(5)

,golden_id,answer,latency
0,1,"{'answer': None, 'status': 'failed', 'error': 'Invalid URL 'timeout': No scheme supplied. Perhaps you meant https://timeout?'}",26.047997
1,2,"{'answer': 'Authors minimize queries and achieve optimal target alignment in black-box generative AI through the development of post-processing algorithms that focus on statistical attribute alignment. These algorithms aim to ensure that the distribution of an attribute of the AI-generated output aligns with a user-specified target. They minimize the expected number of queries to the generator and demonstrate their optimality as the number of requested outputs $ m \to \infty $. The algorithms, such as the Random Demand Coupon (RDC) algorithm, are designed to work within a black-box setting where the user can only query the generative AI model. These post-processing methods complement prompting-based interventions, which may not always be effective due to models not following instructions accurately. The authors also illustrate the effectiveness of their methods through experiments on tasks such as text-to-image generation and geocoded persona generation. URL: https://arxiv.org/pdf/2609.31607', 'status': 'success', 'error': None}",25.567972
2,3,"{'answer': None, 'status': 'failed', 'error': 'Invalid URL 'timeout': No scheme supplied. Perhaps you meant https://timeout?'}",25.262339
3,4,"{'answer': None, 'status': 'failed', 'error': 'Invalid URL 'timeout': No scheme supplied. Perhaps you meant https://timeout?'}",25.287791
4,5,"{'answer': 'Teachers adapt conversational AI use by balancing repair, differentiation, translation, and navigating tensions related to technology, learners, and instruction. They repair failing technology, differentiate for gaps in learners' readiness, translate abstract AI concepts through familiar stories, and balance children's creative agency with learning goals. Teachers constantly navigate these tensions and their adaptive labor bridge the gap for the successful implementation of the AI tool and curriculum. This adaptive work is supported by structured channels such as daily individual and group reflections. URL: https://arxiv.org/pdf/2609.31569', 'status': 'success', 'error': None}",17.194196


My first run with the golden question resulted in error, since the web queries that were generted by the llm were too specific, and the arxive website did returned no results:

```2026-10-05 18:43:55,370 | INFO | user prompt: How does confidence supervision cut tokens by 25% and preserve accuracy Gemma, Qwen, Nemotron, GPT‑OSS?, web query: "confidence supervision token reduction 25% accuracy preservation Gemma Qwen Nemotron GPT-OSS"```

I have tried to resolve this error by prompt engineering and adding a retry query generation. If the first web qury has not found any titles, I prompt the llm to generate a new one and also mark the old query as "bad". In case if the second query has not generated anything, I stop retrying. After trying this approach I had 30% of failed test cases. After examining the failed test cases, I came to the conclusion that some of the generated questions from the golden data set are too specific and not suitable for arxive websearch. My solution to this problem is to let the llm answer the question without using the context. However, to restrict a user from using the RAG model for unrealted to its purpose prompts, I will add a classifier model, that would either let the prompt proceed further in the pipline or not. Only AI, CS related topics can be promt, or other topics that are represented in the arxive web-site.


In [25]:
failed = results[
    results["answer"].apply(lambda x: x["status"] == "failed")
]
print(f"Percentage of failed questions: {len(failed) / len(results) * 100}%")


Percentage of failed questions: 30.0%


In [56]:
logs = pd.read_csv("rag_logs.csv")

retry_then_warning = (
    logs["message"].str.contains("Retry", case=False, na=False)
    & logs["level"].shift(-1).eq("WARNING")
)
retry_then_success = (
    logs["message"].str.contains("Retry", case=False, na=False)
    & ~logs["level"].shift(-1).eq("WARNING")
)

print(f"In {retry_then_warning.sum()} test cases prompt retry did not work")
print(f"In {retry_then_success.sum()} test cases prompt retry worked")



In 15 test cases prompt retry did not work
In 1 test cases prompt retry worked


My solution with retrying worked only for one test case.

In [57]:
pd.set_option("display.max_colwidth", None)
golden_df[golden_df["golden_id"].isin(failed["golden_id"])][["question"]].head(3)

,question
0,"How does confidence supervision cut tokens by 25% and preserve accuracy Gemma, Qwen, Nemotron, GPT‑OSS?"
2,How does static compact doc alone affect issue resolution in real repos vs. source present?
3,"How does OC‑GS refine angles, keep axis, boostPSNR amid uneven rotation, dropped frames, sparse caps?"


In [59]:
pd.set_option("display.max_colwidth", None)
answers = results[results["answer"].apply(lambda x: x["answer"] is not None)][["golden_id", "answer"]].copy()

answers["answer"] = answers["answer"].apply(lambda x: x["answer"])

combined = (golden_df[~golden_df["golden_id"]
            .isin(failed["golden_id"])][["golden_id", "question"]]
            .merge(answers,on="golden_id"))

combined[["question", "answer"]].head(3)

,question,answer
0,"How do authors minimize queries, achieve optimal tgt alignment, post‑process, prompting in black‑box generative AI?","Authors minimize queries and achieve optimal target alignment in black-box generative AI through the development of post-processing algorithms that focus on statistical attribute alignment. These algorithms aim to ensure that the distribution of an attribute of the AI-generated output aligns with a user-specified target. They minimize the expected number of queries to the generator and demonstrate their optimality as the number of requested outputs $ m \to \infty $. The algorithms, such as the Random Demand Coupon (RDC) algorithm, are designed to work within a black-box setting where the user can only query the generative AI model. These post-processing methods complement prompting-based interventions, which may not always be effective due to models not following instructions accurately. The authors also illustrate the effectiveness of their methods through experiments on tasks such as text-to-image generation and geocoded persona generation. \n\nURL: https://arxiv.org/pdf/2609.31607"
1,"How do teachers adapt conv. AI use, balancing repair, differentiation, translation, tech, learner, instr. tensions?","Teachers adapt conversational AI use by balancing repair, differentiation, translation, and navigating tensions related to technology, learners, and instruction. They repair failing technology, differentiate for gaps in learners' readiness, translate abstract AI concepts through familiar stories, and balance children's creative agency with learning goals. Teachers constantly navigate these tensions and their adaptive labor bridge the gap for the successful implementation of the AI tool and curriculum. This adaptive work is supported by structured channels such as daily individual and group reflections. \n\nURL: https://arxiv.org/pdf/2609.31569"
2,How does DeepEdu‑v1’s long‑context engine yield TTFT speedup over vLLM in Vietnam AI tutoring?,"DeepEdu-v1’s long-context inference engine achieves a TTFT (Time-to-First-Token) speedup over vLLM by amortizing token selection from per-sub-chunk to per-cluster granularity. This reduces retrieval calls by 7.7× on long-context retrieval, cutting prefill latency (TTFT) by roughly 35% while maintaining or improving task accuracy. In its deployed configuration, DeepEdu achieves a nearly 2× TTFT speedup over standard vLLM serving. \n\nURL: https://arxiv.org/pdf/2609.31568"


In [65]:
print(f"Mean end-to-end latency: {results["latency"].mean():.2f} sec")

Mean end-to-end latency: 24.36 sec
